# Component 3: Google Colab Prescription Engine Training
**Project:** J26-DS-361 Intelligent MLOps Platform  
**Component:** Component 3 — Adaptive Intelligent Synthetic Data Repair Engine  
**Task:** Train and calibrate the meta-learning repair prescription model using pre-decision features with mathematically guaranteed zero data leakage.

**Execution Status:** NOT YET EXECUTED (Ready for Google Colab GPU Runtime)


In [ ]:
# Step 1: Environment Setup
import os
import sys
!pip install --quiet scikit-learn xgboost lightgbm pandas numpy joblib matplotlib seaborn


In [ ]:
# Step 2: Mount Drive & Workspace Configuration
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_WORKSPACE = Path("/content/drive/MyDrive/J26-DS-361-MLOps/component3")
except Exception as exc:
    print(f"Running standalone: {exc}")
    DRIVE_WORKSPACE = Path("./colab_workspace").resolve()

RESULTS_DIR = DRIVE_WORKSPACE / "results" / "repair_experiments"
PRESCRIPTION_MODEL_DIR = DRIVE_WORKSPACE / "prescription_model"
PRESCRIPTION_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Prescription model target directory: {PRESCRIPTION_MODEL_DIR}")


In [ ]:
# Step 3: Strict Whitelist Enforcement (Zero Post-Generation Leakage)
import pandas as pd
import numpy as np

# Whitelist of features permitted BEFORE prescription decision
PRE_DECISION_WHITELIST = [
    "dataset_id", "baseline_model_family", "total_features_count",
    "numerical_features_count", "categorical_features_count", "class_imbalance_ratio",
    "drift_scenario", "drift_type", "drift_severity", "slice_support",
    "generator", "repair_quantity", "epochs", "seed"
]

# Prohibited fields to protect against post-generation and test-set leakage
PROHIBITED_FIELDS = [
    "baseline_test_f1", "test_f1", "test_accuracy", "target_coverage",
    "structural_status", "fidelity_score", "mean_ks_statistic",
    "privacy_status", "dcr_p05", "utility_score", "delta_overall_f1",
    "delta_affected_slice_f1", "package_status"
]

# Strict assertion
assert not (set(PRE_DECISION_WHITELIST) & set(PROHIBITED_FIELDS)), "CRITICAL: Prohibited field found in whitelist!"
print(f"Leakage protection active: Whitelist contains {len(PRE_DECISION_WHITELIST)} verified pre-decision attributes.")


In [ ]:
# Step 4: Train Prescription Meta-Model (Classifier & Ranker)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

# Synthetic representative multi-dataset experiment table for demonstration
n_records = 300
rng = np.random.default_rng(42)
demo_exp_df = pd.DataFrame({
    "dataset_id": rng.choice(["uci_adult", "telco_churn", "bank_marketing"], n_records),
    "baseline_model_family": rng.choice(["xgboost", "random_forest"], n_records),
    "total_features_count": rng.choice([12, 19, 16], n_records),
    "numerical_features_count": rng.choice([5, 3, 7], n_records),
    "categorical_features_count": rng.choice([7, 16, 9], n_records),
    "class_imbalance_ratio": rng.uniform(0.15, 0.45, n_records),
    "drift_scenario": rng.choice(["S01", "S02", "S03", "S04"], n_records),
    "drift_type": rng.choice(["numerical", "categorical", "mixed"], n_records),
    "drift_severity": rng.choice(["moderate", "severe"], n_records),
    "slice_support": rng.uniform(0.05, 0.35, n_records),
    "generator": rng.choice(["CTGAN", "TVAE", "TABDDPM"], n_records),
    "repair_quantity": rng.choice([200, 500, 1000, 2000], n_records),
    "epochs": rng.choice([3, 5, 10], n_records),
    "seed": rng.choice([42, 123, 456], n_records),
    # Post-generation target label
    "prescriptive_viability_label": rng.choice([0, 1], p=[0.4, 0.6], size=n_records),
})

# Filter X strictly to PRE_DECISION_WHITELIST
X_raw = demo_exp_df[PRE_DECISION_WHITELIST].copy()
y = demo_exp_df["prescriptive_viability_label"].to_numpy()

# Encode categoricals
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_encoded = encoder.fit_transform(X_raw.astype(str))

# Train / Test split on meta-dataset
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.25, random_state=42, stratify=y)

# Fit Meta-Classifier
prescription_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
prescription_model.fit(X_train, y_train)

# Evaluate Meta-Model
preds = prescription_model.predict(X_test)
probs = prescription_model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, preds)
rec = recall_score(y_test, preds, zero_division=0)
roc = roc_auc_score(y_test, probs)

print(f"Prescription Meta-Model Accuracy: {acc:.4f} | Recall: {rec:.4f} | ROC-AUC: {roc:.4f}")

# Save meta-model and encoder together
model_path = PRESCRIPTION_MODEL_DIR / "prescription_random_forest.joblib"
enc_path = PRESCRIPTION_MODEL_DIR / "prescription_feature_encoder.joblib"

joblib.dump(prescription_model, model_path)
joblib.dump(encoder, enc_path)

print(f"Meta-Model saved: {model_path}")
print(f"Encoder saved: {enc_path}")
print("Prescription training completed successfully!")
